In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))  # Add parent directory to path


import pandas as pd
import s3fs
import os
import pandas as pd
from dotenv import load_dotenv
from src.utils.player_utils import PlayerNameMapper


load_dotenv()

fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY"),
    client_kwargs={"region_name": os.getenv("AWS_DEFAULT_REGION", "eu-west-2")}
)


In [ ]:
# List all files in your bucket
bucket = "ucl-ai-soccormon-dataset"
files = fs.ls(bucket)
print("First 5 files in bucket:")
for f in files[:5]:
    print(f)


In [ ]:
top = fs.ls("ucl-ai-soccormon-dataset")
print(top)

In [ ]:
subdirs = fs.ls("ucl-ai-soccormon-dataset/data")
print("First few subdirs:", subdirs)

In [ ]:
# Explore Team A
team_a = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020")
print("Months for Team A:", team_a[:5])

In [ ]:
june = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06")
print("Days in June:", june[:5])

In [ ]:
day1 = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01")
print("Files:", day1)

# Load the first parquet file
df = pd.read_parquet(day1[0], filesystem=fs)
print("Shape:", df.shape)
df.head().to_clipboard()

In [ ]:
from src.utils.player_utils import PlayerNameMapper
from src.utils.day_loader import DayDataLoader

# Assume fs is your s3fs filesystem
mapper = PlayerNameMapper()
loader = DayDataLoader(fs, mapper)

# Load one day
day_path = "ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01"
df_day = loader.load_day(day_path)
print("Loaded shape:", df_day.shape)
print(df_day.head())

# Aggregate to 1s intervals using average
df_agg_mean = loader.aggregate_time(df_day, freq="1s", method="mean")
print("Aggregated mean:", df_agg_mean.head())


In [ ]:
# Third row (nth=2) every 5s
df_nth = loader.aggregate_time(df_day, freq="5s", method="first")

In [ ]:
# Load some parquet file into df
# df = pd.read_parquet(...)

mapper = PlayerNameMapper()

# Apply mapping (auto-updates if new IDs appear)
df_named = mapper.apply_mapping(df)

print(df_named.head())


In [ ]:
df.head().to_clipboard()

In [ ]:
parquet_files = [f for f in fs.ls(bucket) if f.endswith(".parquet")]

print("Found", len(parquet_files), "parquet files")
print("First 3:", parquet_files[:0])

# Load one
df = pd.read_parquet(parquet_files[0], filesystem=fs)
print("Shape:", df.shape)
df.head()

In [ ]:
import pyarrow.parquet as pq

meta = pq.ParquetFile(files[0], filesystem=fs)
print("Num rows:", meta.metadata.num_rows)
print("Num row groups:", meta.num_row_groups)
print("Schema:")
print(meta.schema)

In [ ]:
# Pick a file from the bucket
sample_file = files[0]  # you can also paste a known path
df = pd.read_parquet(files[0], filesystem=fs)
print("Shape:", df.shape)
df.head()


In [ ]:
from src.utils.data_access import S3DataAccess
s3 = S3DataAccess()
df = s3.read_parquet("ucl-ai-soccormon-dataset/path/to/file.parquet")
